In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql import Row

In [0]:
catalog_name="ecommerce"

## brand x category x product 

In [0]:
df_products = spark.table(f"{catalog_name}.silver.slv_product")
df_brands=spark.table(f"{catalog_name}.silver.slv_brands")
df_category=spark.table(f"{catalog_name}.silver.slv_category")

In [0]:
df_products.createOrReplaceTempView("v_products")
df_brands.createOrReplaceTempView("v_brands")
df_category.createOrReplaceTempView("v_category")

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")

In [0]:
%sql
-- build brands x category mapping amd writing Gold table
CREATE OR REPLACE TABLE gold.gld_dim_products AS 

WITH brand_categories AS(
    SELECT 
    b.brand_name,
    b.brand_code,
    c.category_code,
    c.category_name
    FROM v_brands b
    INNER JOIN v_category c
    ON b.category_code = c.category_code
)

SELECT
p.product_id,
p.sku,
p.category_code,
COALESCE(bc.category_name, 'NOT AVAILABLE') AS category_name,
p.brand_code,
COALESCE(bc.brand_name, 'NOT AVAILABLE') AS brand_name,
p.color,
p.size,
p.material,
p.weight_grams,
p.length_cm,
p.width_cm,
p.height_cm,
p.rating_count,
p._source_file,
p._ingested_at
From v_products p 
LEFT JOIN brand_categories bc
ON p.brand_code = bc.brand_code;

## customers table

In [0]:
# India_states
india_region ={
    "MH":"West","GJ":"West","RJ":"West",
    "KA":"South","TN":"South","KL":"South","AP":"South","TS":"South",
    "DL":"North","UP":"North","WB":"East"
}

# Australia states
autralia_region ={
    "QLD":"NorthEast","NSW":"East","VIC":"SouthEast",
    "WA":"West"
}

# United Kingdom states
UK_region ={
    "ENG":"England","WLS":"Wales","NIR":"Northen Ireland","SCT":"Scotland"
}

# United States states
US_region ={
    "MA" : "NorthEast", "FL":"South","NJ":"NorthEast","CA":"West",
    "NY":"NorthEast","TX":"South"
}

# UAE states
uae_region ={
    "AUH":"Abu Dhabi", "DU":"Dubai", "SHJ":"Sharjah"
}

# Singapore states
singapore_region ={
    "SG":"Singapore"
}

# Canada States
canada_region ={
   "BC":"West", "AB":"West","ON":"East", "QC":"East","NS":"East","IL":"Other"
}

# combine into master dictionary
country_state_map={
    "India": india_region,
    "Australia":autralia_region,
    "United Kingdom":UK_region,
    "United States":US_region,
    "UAE":uae_region,
    "Singapore":singapore_region,
    "Canada":canada_region
}

In [0]:
# 1 Flatten country_state_map into a list of Rows
rows =[]
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))
rows[:10]

In [0]:
df_region_mapping= spark.createDataFrame(rows)

# Optional : show mapping 
df_region_mapping.show(truncate=False)

In [0]:


df_silver= spark.table(f'{catalog_name}.silver.slv_customers')
df_gold = df_silver.join(df_region_mapping, on=['country','state'],how='left')

df_gold=df_gold.fillna({'region':'Other'})

In [0]:
display(df_gold)

In [0]:
df_gold.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.gold.gld_dim_customer")

## Calender/date

In [0]:
df_silver=spark.table(f'{catalog_name}.silver.slv_date')
display(df_silver.limit(5))

In [0]:
df_gold= df_silver.withColumn("date_id",F.date_format(F.col("date"),"yyyyMMdd").cast("int"))

# Add month name (eg., 'January')
df_gold=df_gold.withColumn('month_name',F.date_format(F.col("date"),"MMMM"))

# Add is_weekend
df_gold=df_gold.withColumn(
    'is_weekend',
    F.when(F.col("day_name").isin("Saturday","Sunday"),1).otherwise(0)
    )

display(df_gold.limit(5))

In [0]:
desired_columns_order=['date_id','date','year','month_name','day_name','is_weekend','quarter','week','_source_file','_ingested_at']
df_gold=df_gold.select(desired_columns_order)
df_gold.show(10)

In [0]:
df_gold.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.gold.gld_dim_date")